# 🧠 Stroke Network Isochrone Analysis – Salta (Argentina)


This notebook (n.1) analyzes geographic accessibility to stroke care (ACV)
using network-based isochrones.

***Main features:***
- Road network extraction using OSMnx
- Travel-time estimation with realistic ambulance speeds
- Isochrone generation (1h, 2h, 4h)
- Population coverage analysis
- Risk modeling using socioeconomic indicators
- Interactive maps with Folium

***Author: Martin Bielke***

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
from folium.plugins import HeatMap
from sklearn.preprocessing import MinMaxScaler

import branca.colormap as cm
import time

# Selenium opcional
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False


# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------
ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.overpass_settings = '[out:json][timeout:180]'

BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

GRAPH_FILE = os.path.join(BASE_DIR, "salta_graph_local.graphml")
EXCEL_POB = os.path.join(BASE_DIR, "data", "poblacion.xlsx")  
    
COL_POB = "Población total (en hogares familiares)."
COL_LAT = "Latitud del centroide"
COL_LON = "Longitud del centroide"

hospitals = {
    "San Bernardo": (-24.782, -65.412),
    "Metan": (-25.483, -64.967),
    "Tartagal": (-22.516, -63.801),
    "Cafayate": (-26.073, -65.976),
    "Rosario": (-25.803, -64.970),
    "Oran": (-23.132, -64.324)
}

trip_times = [60, 120, 240]
# CAMBIO: color de 4 horas ahora es azul oscuro (#0033cc)
iso_colors_hex = ["#ff4d4d", "#ffaa00", "#0033cc"]   # rojo, naranja, azul oscuro
iso_names = ["1 hora", "2 horas", "4 horas"]
LOCAL_CRS = "EPSG:32720"
#LOCAL_CRS = "EPSG:32720"  # WGS 84 / UTM zona 20S

OUT_DIR_HTML = os.path.join(BASE_DIR, "output", "html")
OUT_DIR_PNG = os.path.join(BASE_DIR, "output", "png")
os.makedirs(OUT_DIR_HTML, exist_ok=True)
os.makedirs(OUT_DIR_PNG, exist_ok=True)

# ------------------------------------------------------------
# 0. CARGA DE POBLACIÓN (antes de isócronas)
# ------------------------------------------------------------
print("\nCargando datos censales...")
df = pd.read_excel(EXCEL_POB)
gdf_pop_wgs84 = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[COL_LON], df[COL_LAT]),
    crs="EPSG:4326"
)
gdf_pop_local = gdf_pop_wgs84.to_crs(LOCAL_CRS)
total_poblacion = gdf_pop_local[COL_POB].sum()
print(f"Población total provincial: {total_poblacion:,.0f} hab")

bounds = gdf_pop_wgs84.total_bounds
sw = [bounds[1], bounds[0]]
ne = [bounds[3], bounds[2]]

# ------------------------------------------------------------
# 1. CARGA DEL GRAFO
# ------------------------------------------------------------
if os.path.exists(GRAPH_FILE):
    print("Cargando grafo guardado...")
    G = ox.load_graphml(GRAPH_FILE)
else:
    print("Descargando grafos locales...")
    graphs = []
    DIST = 80000 
    for name, (lat, lon) in hospitals.items():
        print(f" → {name}")
        G_local = ox.graph_from_point((lat, lon), dist=DIST, network_type='drive')
        graphs.append(G_local)
    G = nx.compose_all(graphs)
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    ox.save_graphml(G, GRAPH_FILE)
    print("Grafo guardado.")

# ------------------------------------------------------------
# --- AJUSTE DE VELOCIDADES (más realistas para ambulancias en Salta) ---
# Adjust edge speeds to reflect realistic ambulance travel conditions
# in heterogeneous terrains (urban vs rural areas in Salta)

print("Ajustando velocidades a valores realistas para ambulancias...")
for u, v, data in G.edges(data=True):
    highway = data.get('highway', 'unknown')
    if isinstance(highway, list):
        highway = highway[0]
    # Velocidades en km/h (conservadoras)
    if highway in ['motorway', 'trunk']:
        speed = 80
    elif highway in ['primary', 'secondary']:
        speed = 60
    elif highway == 'tertiary':
        speed = 50
    elif highway == 'residential':
        speed = 30
    elif highway == 'track':
        speed = 20
    else:
        speed = 40
    data['speed_kph'] = speed
    data['travel_time'] = data['length'] / (speed / 3.6)
print("Velocidades ajustadas.")
print(f"Grafo listo: {len(G.nodes)} nodos, {len(G.edges)} aristas")

# ------------------------------------------------------------
# 2. FUNCIÓN DE CÁLCULO DE ISÓCRONAS
# ------------------------------------------------------------
def get_isochrones_for_hospital(G, center_coords, trip_times):
    """
    Calcula polígonos de isócronas basados en la red vial (Network-based).
    Buffer reducido a 150 m para evitar sobreestimación.
    
    Parameters:
    - G: road network graph (NetworkX)
    - center_coords: (lat, lon)
    - trip_times: list of travel times in minutes

    Returns:
    - dict: {time: polygon}

    Method:
    Uses Dijkstra shortest paths based on travel_time and applies
    a 150m buffer to approximate reachable areas.
    """
    
    center_node = ox.nearest_nodes(G, X=center_coords[1], Y=center_coords[0])
    times_sec = [t * 60 for t in trip_times]
    travel_times = nx.single_source_dijkstra_path_length(G, center_node, weight='travel_time')
    
    nodes, edges = ox.graph_to_gdfs(G)
    edges_proj = edges.to_crs(LOCAL_CRS)

    isochrones = {}
    for t_sec, t_min in zip(times_sec, trip_times):
        reachable_nodes = [n for n, tt in travel_times.items() if tt <= t_sec]
        if not reachable_nodes:
            continue
        reachable_edges = edges_proj[
            edges_proj.index.get_level_values(0).isin(reachable_nodes) & 
            edges_proj.index.get_level_values(1).isin(reachable_nodes)
        ]
        if reachable_edges.empty:
            continue
        # Buffer reducido a 150 m
        poly = reachable_edges.geometry.buffer(150).unary_union
        if not poly.is_valid:
            poly = poly.buffer(0)
        isochrones[t_min] = poly
    return isochrones

# ------------------------------------------------------------
# 3. CÁLCULO DE ISÓCRONAS
# ------------------------------------------------------------
print("\nCalculando isócronas individuales...")
isochrones_by_hospital = {}
all_polygons_by_time = {t: [] for t in trip_times}

for name, coords in hospitals.items():
    print(f"  {name}")
    polys = get_isochrones_for_hospital(G, coords, trip_times)
    isochrones_by_hospital[name] = polys
    for t, poly in polys.items():
        all_polygons_by_time[t].append(poly)

union_polygons_local = {}
union_polygons_wgs84 = {}
for t in trip_times:
    if all_polygons_by_time[t]:
        from shapely.ops import unary_union
        cobertura_total = unary_union(all_polygons_by_time[t])
        cobertura_total = cobertura_total.simplify(50)
        cobertura_total = cobertura_total.buffer(0)
        
        union_polygons_local[t] = cobertura_total
        union_polygons_wgs84[t] = gpd.GeoSeries([cobertura_total], crs=LOCAL_CRS).to_crs("EPSG:4326").iloc[0]
        
        area_km2 = union_polygons_local[t].area / 1e6
        print(f"Cobertura combinada {t//60}h: área = {area_km2:.1f} km²")
    else:
        union_polygons_local[t] = None
        union_polygons_wgs84[t] = None        

# ------------------------------------------------------------
# 4. ESTADÍSTICAS POBLACIONALES DE COBERTURA
# ------------------------------------------------------------
stats_rows = []
for name, polys in isochrones_by_hospital.items():
    row = {"Hospital": name}
    for t in trip_times:
        if t in polys:
            pop = gdf_pop_local[gdf_pop_local.geometry.within(polys[t])][COL_POB].sum()
            row[f"Población_{t//60}h"] = int(pop)
        else:
            row[f"Población_{t//60}h"] = 0
    stats_rows.append(row)

df_hospital_stats = pd.DataFrame(stats_rows).set_index("Hospital")
print("\n=== Cobertura poblacional por hospital ===")
print(df_hospital_stats.to_string())

union_coverage_pop = {}
for t in trip_times:
    if union_polygons_local[t] is not None:
        covered = gdf_pop_local[gdf_pop_local.geometry.within(union_polygons_local[t])]
        union_coverage_pop[t] = covered[COL_POB].sum()
    else:
        union_coverage_pop[t] = 0

total_cubierta_4h = union_coverage_pop[240]
porcentaje_cubierto = (total_cubierta_4h / total_poblacion) * 100
area_total_km2 = union_polygons_local[240].area / 1e6 if union_polygons_local[240] else 0

print("\n=== Resumen red completa ===")
print(f"Población cubierta (≤4h): {total_cubierta_4h:,.0f} ({porcentaje_cubierto:.1f}%)")
print(f"Área cubierta: {area_total_km2:,.0f} km²")

# Guardar CSV
df_hospital_stats.to_csv(os.path.join(OUT_DIR_HTML, "cobertura_por_hospital.csv"))
summary_df = pd.DataFrame({
    "Indicador": ["Población total", "Cubierta (≤4h)", "Porcentaje", "Área (km²)"],
    "Valor": [f"{total_poblacion:,.0f}", f"{total_cubierta_4h:,.0f}", f"{porcentaje_cubierto:.1f}%", f"{area_total_km2:,.0f}"]
})
summary_df.to_csv(os.path.join(OUT_DIR_HTML, "resumen_red_completa.csv"), index=False)

# ------------------------------------------------------------
# 5. CARGA DE INDICADORES DE RIESGO (CSVs) y CÁLCULO DE RISK SCORE
# ------------------------------------------------------------
print("\nCargando indicadores de riesgo desde CSVs...")

ruta2 = os.path.join(BASE_DIR, "data/indicadores_2.csv")
ruta3 = os.path.join(BASE_DIR, "data/indicadores_3.csv")
ruta4 = os.path.join(BASE_DIR, "data/indicadores_4.csv")


def load_and_clean_csv(path):
    df = pd.read_csv(path, encoding='utf-8', sep=';')
    df.columns = (df.columns.str.strip()
                  .str.lower()
                  .str.normalize('NFKD')
                  .str.encode('ascii', errors='ignore')
                  .str.decode('utf-8')
                  .str.replace(' ', '_')
                  .str.replace('á','a').str.replace('é','e').str.replace('í','i')
                  .str.replace('ó','o').str.replace('ú','u').str.replace('ñ','n')
                  .str.replace(r'\.$','', regex=True))
    codigo_cols = [col for col in df.columns if 'codigo' in col and 'radio' in col]
    if len(codigo_cols) == 0:
        raise KeyError("No se encontró columna de código de radio")
    codigo_col = codigo_cols[0]
    df.rename(columns={codigo_col: 'codigo_radio'}, inplace=True)
    for col in df.columns:
        if col != 'codigo_radio':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.loc[:, ~df.columns.duplicated()]
    return df

df2 = load_and_clean_csv(ruta2)
df3 = load_and_clean_csv(ruta3)
df4 = load_and_clean_csv(ruta4)

gdf_temp = df2.merge(df3, on='codigo_radio', how='left', suffixes=('_2', '_3'))
gdf_indicadores = gdf_temp.merge(df4, on='codigo_radio', how='left', suffixes=('', '_4'))

if 'codigo_radio' not in gdf_pop_local.columns:
    for col in gdf_pop_local.columns:
        if 'radio' in col.lower() or 'codigo' in col.lower():
            gdf_pop_local = gdf_pop_local.rename(columns={col: 'codigo_radio'})
            break
    else:
        if 'codigo_de_radio' in df.columns:
            gdf_pop_local['codigo_radio'] = df['codigo_de_radio'].values
        elif 'codigo_radio' in df.columns:
            gdf_pop_local['codigo_radio'] = df['codigo_radio'].values

gdf_modelo = gdf_indicadores.merge(gdf_pop_local[['codigo_radio', 'geometry']], on='codigo_radio', how='left')
gdf_modelo = gpd.GeoDataFrame(gdf_modelo, geometry='geometry', crs=gdf_pop_local.crs)

cols_riesgo_base = [
    'hogares_con_hacinamiento_critico_(mas_de_3_personas_por_cuarto)',
    'hogares_sin_cloaca',
    'hogares_sin_agua_para_beber_y_cocinar_proveniente_de_red_publica',
    'hogares_con_computadora',
    'hogares_con_telefono_celular',
    'poblacion_de_70_anos_y_mas',
    'solo_salud_publica',
    'poblacion_de_18_y_mas_con_primaria_incompleta_o_menos'
]

for col in cols_riesgo_base:
    if col not in gdf_modelo.columns:
        for suf in ['_3', '_4', '_2']:
            col_suf = f"{col}{suf}"
            if col_suf in gdf_modelo.columns:
                gdf_modelo[col] = gdf_modelo[col_suf]
                break

existing_risk_cols = [col for col in cols_riesgo_base if col in gdf_modelo.columns]
missing = [col for col in cols_riesgo_base if col not in existing_risk_cols]
if missing:
    print("⚠ Columnas de riesgo no encontradas (serán ignoradas):", missing)

if 'hogares_con_computadora' in existing_risk_cols:
    gdf_modelo['hogares_con_computadora_inv'] = 1 - gdf_modelo['hogares_con_computadora'].fillna(0)/100
    existing_risk_cols.append('hogares_con_computadora_inv')
    existing_risk_cols.remove('hogares_con_computadora')
if 'hogares_con_telefono_celular' in existing_risk_cols:
    gdf_modelo['hogares_con_telefono_celular_inv'] = 1 - gdf_modelo['hogares_con_telefono_celular'].fillna(0)/100
    existing_risk_cols.append('hogares_con_telefono_celular_inv')
    existing_risk_cols.remove('hogares_con_telefono_celular')

if len(existing_risk_cols) > 0:
    scaler = MinMaxScaler()
    X = gdf_modelo[existing_risk_cols].fillna(0)
    risk_normalized = scaler.fit_transform(X)
    gdf_modelo['risk_score_norm'] = risk_normalized.mean(axis=1)
    print("✅ Risk_score calculado correctamente.")
else:
    print("❌ No hay columnas de riesgo, no se puede calcular risk_score.")
    gdf_modelo['risk_score_norm'] = np.nan

# ------------------------------------------------------------
# 6. FUNCIÓN PARA GUARDAR MAPAS (HTML + PNG)
# ------------------------------------------------------------
def html_to_png(html_path, png_path, width=1200, height=900):
    if not SELENIUM_AVAILABLE:
        return
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument(f"--window-size={width},{height}")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        driver.get("file://" + os.path.abspath(html_path))
        time.sleep(4)
        driver.save_screenshot(png_path)
        driver.quit()
        print(f"   📸 PNG guardado: {png_path}")
    except Exception as e:
        print(f"   ⚠️ PNG no generado: {e}")

def save_map(m, name, also_png=True):
    html_path = os.path.join(OUT_DIR_HTML, f"{name}.html")
    m.save(html_path)
    print(f"✅ Mapa HTML: {html_path}")
    if also_png:
        png_path = os.path.join(OUT_DIR_PNG, f"{name}.png")
        html_to_png(html_path, png_path)

# ------------------------------------------------------------
# 7. MAPA 1: COBERTURA COMBINADA
# ------------------------------------------------------------
m1 = folium.Map(tiles='CartoDB positron')
m1.fit_bounds([sw, ne])

heat_data = [[row.geometry.y, row.geometry.x, row[COL_POB]] for _, row in gdf_pop_wgs84.iterrows()]
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m1)

for i, t in enumerate(trip_times):
    if union_polygons_wgs84[t] is None:
        continue
    pop = union_coverage_pop[t]
    folium.GeoJson(
        union_polygons_wgs84[t].__geo_interface__,
        style_function=lambda x, c=iso_colors_hex[i]: {'fillColor': c, 'color': c, 'weight': 1.5, 'fillOpacity': 0.2},
        tooltip=f"{t//60}h: {pop:,.0f} personas",
        name=iso_names[i]
    ).add_to(m1)

for name, (lat, lon) in hospitals.items():
    folium.Marker(location=[lat, lon], popup=name, tooltip=name,
                  icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m1)

# Leyenda actualizada: 4 horas ahora es azul
legend1_html = f"""
<div style="position: fixed; bottom: 30px; left: 30px; width: 300px; 
     background-color: white; border:2px solid grey; border-radius:8px; 
     padding: 10px; font-size: 13px; z-index: 9999; 
     box-shadow: 3px 3px 6px rgba(0,0,0,0.3);">
<h4 style="margin-top:0; text-align:center;">Isocronas y cobertura poblacional</h4>
<b>🚑 Tiempo de viaje en ambulancia:</b><br>
<span style="display:inline-block; width:20px; height:12px; background:#ff4d4d;"></span> 1 hora<br>
<span style="display:inline-block; width:20px; height:12px; background:#ffaa00;"></span> 2 horas<br>
<span style="display:inline-block; width:20px; height:12px; background:#0033cc;"></span> 4 horas<br>
<hr>
<b>🔥 Mapa de calor (población):</b><br>
<div style="width:100%; height:20px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
<span style="float:left;">Menor población</span><span style="float:right;">Mayor población</span><br clear="all">
<hr>
<b>🏥 Hospitales:</b> Icono “+” negro<br>
<b>📊 Población total cubierta (≤4h):</b> {total_cubierta_4h:,.0f} personas ({porcentaje_cubierto:.1f}%)<br>
<small>Los tooltips muestran población dentro de cada isócrona.</small>
</div>
"""
m1.get_root().html.add_child(folium.Element(legend1_html))
folium.LayerControl().add_to(m1)
save_map(m1, "mapa_cobertura_combinada", also_png=True)

# ------------------------------------------------------------
# 8. MAPA 2: SOLO DENSIDAD POBLACIONAL
# ------------------------------------------------------------
m2 = folium.Map(tiles='CartoDB positron')
m2.fit_bounds([sw, ne])
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m2)

legend2_html = f"""
<div style="position: fixed; bottom: 30px; left: 30px; width: 280px; 
     background-color: white; border:2px solid grey; border-radius:8px; 
     padding: 10px; font-size: 13px; z-index: 9999;">
<h4 style="margin-top:0;">Densidad poblacional - Salta (Censo 2022)</h4>
<div style="width:100%; height:20px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
<span style="float:left;">Baja densidad</span><span style="float:right;">Alta densidad</span><br clear="all">
<hr>
<b>Población total provincial:</b> {total_poblacion:,.0f} personas<br>
<small>Cada radio censal ponderado por su población.</small>
</div>
"""
m2.get_root().html.add_child(folium.Element(legend2_html))
save_map(m2, "mapa_densidad_poblacional", also_png=True)

# ------------------------------------------------------------
# 9. MAPAS INDIVIDUALES POR HOSPITAL
# ------------------------------------------------------------
for hosp_name, coords in hospitals.items():
    m = folium.Map(tiles='CartoDB positron')
    m.fit_bounds([sw, ne])
    HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m)

    polys_local = isochrones_by_hospital.get(hosp_name, {})
    for i, t in enumerate(trip_times):
        if t not in polys_local:
            continue
        poly_wgs84 = gpd.GeoSeries([polys_local[t]], crs=LOCAL_CRS).to_crs("EPSG:4326").iloc[0]
        pop_in = gdf_pop_local[gdf_pop_local.geometry.within(polys_local[t])][COL_POB].sum()
        folium.GeoJson(
            poly_wgs84.__geo_interface__,
            style_function=lambda x, c=iso_colors_hex[i]: {'fillColor': c, 'color': c, 'weight': 1.5, 'fillOpacity': 0.2},
            tooltip=f"{t//60}h: {pop_in:,.0f} personas",
            name=iso_names[i]
        ).add_to(m)

    folium.Marker(location=coords, popup=hosp_name, tooltip=hosp_name,
                  icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m)

    # Leyenda individual con azul para 4h
    legend_hosp_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: 280px; 
         background-color: white; border:2px solid grey; border-radius:8px; padding: 10px; font-size: 13px; z-index: 9999;">
    <b>{hosp_name}</b><br>
    <span style="background:#ff4d4d; width:20px;height:12px;display:inline-block;"></span> 1h &nbsp;
    <span style="background:#ffaa00; width:20px;height:12px;display:inline-block;"></span> 2h &nbsp;
    <span style="background:#0033cc; width:20px;height:12px;display:inline-block;"></span> 4h<br>
    <div style="width:100%; height:15px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
    <small>Mapa de calor: densidad poblacional. Tooltips muestran población por isócrona.</small>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_hosp_html))
    folium.LayerControl().add_to(m)
    save_map(m, f"mapa_{hosp_name.replace(' ', '_')}", also_png=True)


# ------------------------------------------------------------
# 10. MAPA DE RIESGO FUERA DE COBERTURA (2h) - coloreado por riesgo
# ------------------------------------------------------------
print("\n--- Generando mapa de riesgo fuera de cobertura 2h ---")

# ------------------- 2. FILTRAR RADIOS FUERA DE COBERTURA 2h -------------------
tiempo_cluster = 120  # minutos
if union_polygons_local.get(tiempo_cluster) is None:
    print(f"No hay polígono de cobertura para {tiempo_cluster//60}h.")
else:
    poly_local = union_polygons_local[tiempo_cluster]
    fuera = gdf_pop_local[~gdf_pop_local.geometry.within(poly_local)].copy()
    print(f"Radios fuera de cobertura (sin filtrar por población): {len(fuera)} de {len(gdf_pop_local)}")
    print(f"Población fuera (sin filtrar): {fuera[COL_POB].sum():,.0f} hab")

    if 'codigo_radio' not in fuera.columns:
        for col in fuera.columns:
            if 'radio' in col.lower() or 'codigo' in col.lower():
                fuera = fuera.rename(columns={col: 'codigo_radio'})
                print(f"Columna '{col}' renombrada a 'codigo_radio'")
                break
        else:
            raise KeyError("No se encontró columna de código de radio en 'fuera'.")

    # Normalizar claves para el merge
    fuera['codigo_radio'] = fuera['codigo_radio'].astype(str).str.strip()
    gdf_modelo['codigo_radio'] = gdf_modelo['codigo_radio'].astype(str).str.strip()

    fuera = fuera.merge(gdf_modelo[['codigo_radio', 'risk_score_norm']], on='codigo_radio', how='left')
    fuera = fuera.dropna(subset=['risk_score_norm'])

    # +++ FILTRO: eliminar radios con población 0 +++
    fuera = fuera[fuera[COL_POB] > 0].copy()

    print(f"Radios fuera de cobertura con población > 0 y riesgo válido: {len(fuera)}")
    print(f"Población fuera de cobertura (excluyendo radios sin habitantes): {fuera[COL_POB].sum():,.0f} hab")

    if len(fuera) == 0:
        print("⚠ ADVERTENCIA: No hay radios fuera de cobertura con población positiva y riesgo válido.")
    else:
        print(f"Rango de riesgo: min={fuera['risk_score_norm'].min():.3f}, max={fuera['risk_score_norm'].max():.3f}")

    fuera_wgs84 = fuera.to_crs(epsg=4326)

    m_fuera = folium.Map(location=[-24.782, -65.423], zoom_start=8, tiles='CartoDB positron')

    if union_polygons_wgs84.get(tiempo_cluster):
        folium.GeoJson(
            union_polygons_wgs84[tiempo_cluster].__geo_interface__,
            style_function=lambda x: {'fillColor': '#4dff4d', 'color': '#4dff4d', 'weight': 1, 'fillOpacity': 0.1},
            tooltip="Cobertura 2h"
        ).add_to(m_fuera)

    try:
        from branca.colormap import linear
        risk_colormap = linear.YlOrRd_09.scale(0, 1)
        risk_colormap.caption = 'Riesgo normalizado'
    except (ImportError, NameError):
        def risk_colormap(x):
            r = 255
            g = int(255 * (1 - x))
            b = 0
            return f'#{r:02x}{g:02x}{b:02x}'
        print("Usando colormap manual (fallback)")

    if len(fuera) > 0:
        for idx, row in fuera_wgs84.iterrows():
            try:
                color = risk_colormap(row['risk_score_norm'])
            except Exception:
                color = 'red'
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=4,
                color=color,
                fill=True,
                fill_opacity=0.8,
                popup=f"Radio: {row['codigo_radio']}<br>Población: {row[COL_POB]:,.0f}<br>Riesgo: {row['risk_score_norm']:.2f}"
            ).add_to(m_fuera)

        if hasattr(risk_colormap, 'add_to'):
            risk_colormap.add_to(m_fuera)

    riesgo_promedio = fuera['risk_score_norm'].mean() if len(fuera) > 0 else 0
    leyenda_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: 300px; background-color: white; border:2px solid grey; border-radius:8px; padding: 10px; font-size: 12px; z-index: 9999;">
    <b>Radios fuera de cobertura 2h</b><br>
    Población total: {fuera[COL_POB].sum():,.0f} hab<br>
    N° de radios: {len(fuera)}<br>
    Riesgo promedio: {riesgo_promedio:.2f}<br>
    <hr>
    <span style="background:#ffffb2; width:20px;height:12px;display:inline-block;"></span> Bajo riesgo<br>
    <span style="background:#fd8d3c; width:20px;height:12px;display:inline-block;"></span> Medio riesgo<br>
    <span style="background:#bd0026; width:20px;height:12px;display:inline-block;"></span> Alto riesgo<br>
    <small>El color de cada punto indica su nivel de riesgo (escala amarillo-rojo).<br>Se excluyen radios con población = 0.</small>
    </div>
    """
    m_fuera.get_root().html.add_child(folium.Element(leyenda_html))

    map_file = os.path.join(OUT_DIR_HTML, "mapa_riesgo_fuera_cobertura_2h.html")
    m_fuera.save(map_file)
    print(f"✅ Mapa de riesgo fuera de cobertura guardado: {map_file}")

    if SELENIUM_AVAILABLE:
        png_path = os.path.join(OUT_DIR_PNG, "mapa_riesgo_fuera_cobertura_2h.png")
        html_to_png(map_file, png_path)
    
# ------------------------------------------------------------
# 11. MAPA DE COBERTURA 4h con heatmap acotado
# ------------------------------------------------------------
if union_polygons_wgs84[240] is not None:
    m_punto2 = folium.Map(tiles='CartoDB positron')
    bounds_4h = union_polygons_wgs84[240].bounds
    m_punto2.fit_bounds([[bounds_4h[1], bounds_4h[0]], [bounds_4h[3], bounds_4h[2]]])

    gdf_pop_4h = gdf_pop_local[gdf_pop_local.geometry.within(union_polygons_local[240])].copy()
    gdf_pop_4h_wgs84 = gdf_pop_4h.to_crs("EPSG:4326")
    heat_data_4h = [[row.geometry.y, row.geometry.x, row[COL_POB]] for _, row in gdf_pop_4h_wgs84.iterrows()]
    HeatMap(heat_data_4h, radius=15, blur=10, min_opacity=0.3,
            name="Densidad poblacional (dentro de 4h)").add_to(m_punto2)

    folium.GeoJson(
        union_polygons_wgs84[240].__geo_interface__,
        style_function=lambda x: {'fillColor': '#0033cc', 'color': '#0033cc', 'weight': 2, 'fillOpacity': 0.2},
        tooltip=f"4 horas: {union_coverage_pop[240]:,.0f} personas",
        name="Cobertura 4h"
    ).add_to(m_punto2)

    for name, (lat, lon) in hospitals.items():
        folium.Marker(location=[lat, lon], popup=name, tooltip=name,
                      icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m_punto2)

    legend_punto2 = f"""
    <div style="position: fixed; bottom: 30px; right: 30px; width: 280px; 
         background-color: white; border:2px solid grey; border-radius:8px; 
         padding: 10px; font-size: 12px; z-index: 9999; 
         box-shadow: 3px 3px 6px rgba(0,0,0,0.3);">
    <b>🗺️ Cobertura de 4 horas – Red de ACV</b><br>
    <span style="background:#0033cc; width:20px;height:12px;display:inline-block;"></span> Área alcanzable en ≤4h<br>
    <div style="width:100%; height:15px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
    <span style="float:left;">Baja densidad</span><span style="float:right;">Alta densidad</span><br clear="all">
    <b>🏥 Hospitales:</b> + negro<br>
    <b>👥 Población dentro del área:</b> {union_coverage_pop[240]:,.0f} hab ({porcentaje_cubierto:.1f}%)<br>
    <small>El mapa de calor solo muestra población dentro de la cobertura de 4h.</small>
    </div>
    """
    m_punto2.get_root().html.add_child(folium.Element(legend_punto2))
    folium.LayerControl().add_to(m_punto2)
    save_map(m_punto2, "mapa_cobertura_4h_con_heatmap_acotado", also_png=True)

# ------------------------------------------------------------
# 12. MAPAS DE COBERTURA POR TIEMPO (1h, 2h, 4h)
# ------------------------------------------------------------
for i, t in enumerate(trip_times):
    if union_polygons_wgs84[t] is None:
        continue
    m = folium.Map(tiles='CartoDB positron')
    m.fit_bounds([sw, ne])
    HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m)
    pop_in = union_coverage_pop[t]
    folium.GeoJson(
        union_polygons_wgs84[t].__geo_interface__,
        style_function=lambda x, c=iso_colors_hex[i]: {'fillColor': c, 'color': c, 'weight': 2, 'fillOpacity': 0.3},
        tooltip=f"{t//60}h: {pop_in:,.0f} personas",
        name=f"{t//60}h"
    ).add_to(m)
    for name, (lat, lon) in hospitals.items():
        folium.Marker(location=[lat, lon], popup=name, icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m)

    legend_time_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: 260px; 
         background-color: white; border:2px solid grey; border-radius:8px; 
         padding: 10px; font-size: 13px; z-index: 9999;">
    <b>Cobertura de {t//60} hora(s)</b><br>
    <span style="background:{iso_colors_hex[i]}; width:20px;height:12px;display:inline-block;"></span> Isócrona {t//60}h<br>
    <b>Población dentro del área:</b> {pop_in:,.0f} hab<br>
    <div style="width:100%; height:15px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
    <small>Mapa de calor: densidad poblacional. Los tooltips muestran la población exacta.</small>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_time_html))
    save_map(m, f"mapa_cobertura_{t//60}h", also_png=True)

print("\n" + "="*60)
print("✅ ¡PROCESO COMPLETADO EXITOSAMENTE!")
print(f"📁 HTMLs interactivos: {os.path.abspath(OUT_DIR_HTML)}")
print(f"🖼️  Imágenes PNG: {OUT_DIR_PNG}")
print("="*60)

# Geographic accessibility to stroke care (ACV) using convex hull isochrones

This notebook (n.2) analyzes geographic accessibility to stroke care using isochrones based on the **convex hull of reachable nodes**. Unlike Notebook 1, it uses default OSM travel speeds (no ambulance-specific adjustments) and generates isochrones as convex polygons around reachable nodes rather than buffered edges.

**Main features:**

- Road network extraction using OSMnx
- Travel-time estimation using default OSM speeds (standard driving)
- Isochrone generation via **convex hull** of reachable nodes (1h, 2h, 4h)
- Population coverage analysis
- Risk modeling using socioeconomic indicators (same as Notebook 1)
- Interactive maps with Folium
- Automatic export to HTML and PNG (optional Selenium)

**Key differences from Notebook 1:**

- Isochrone method: `convex_hull` of nodes (faster, less precise in rural areas)
- Speeds: default OSM values (no ambulance prioritization)
- Colors: 4‑hour isochrone in green (`#4dff4d`) instead of dark blue
- Legend: refers to "car travel time" (not ambulance)

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
from folium.plugins import HeatMap
from sklearn.preprocessing import MinMaxScaler

import branca.colormap as cm
import time

# Selenium opcional
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

# ------------------------------------------------------------
# CONFIGURACIÓN (con BASE_DIR y rutas relativas)
# ------------------------------------------------------------
ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.overpass_settings = '[out:json][timeout:180]'

BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
GRAPH_FILE = os.path.join(BASE_DIR, "salta_graph_local.graphml")
EXCEL_POB = os.path.join(BASE_DIR, "data", "poblacion.xlsx")
COL_POB = "Población total (en hogares familiares)."
COL_LAT = "Latitud del centroide"
COL_LON = "Longitud del centroide"

hospitals = {
    "San Bernardo": (-24.782, -65.412),
    "Metan": (-25.483, -64.967),
    "Tartagal": (-22.516, -63.801),
    "Cafayate": (-26.073, -65.976),
    "Rosario": (-25.803, -64.970),
    "Oran": (-23.132, -64.324)
}

trip_times = [60, 120, 240]
iso_colors_hex = ["#ff4d4d", "#ffaa00", "#4dff4d"]   # se mantiene verde para 4h
iso_names = ["1 hora", "2 horas", "4 horas"]
LOCAL_CRS = "EPSG:32720"

# Directorios de salida unificados (como en el primer script)
OUT_DIR_HTML = os.path.join(BASE_DIR, "output", "html")
OUT_DIR_PNG = os.path.join(BASE_DIR, "output", "png")
os.makedirs(OUT_DIR_HTML, exist_ok=True)
os.makedirs(OUT_DIR_PNG, exist_ok=True)

# ------------------------------------------------------------
# 0. CARGA DE POBLACIÓN (antes de isócronas)
# ------------------------------------------------------------
print("\nCargando datos censales...")
df = pd.read_excel(EXCEL_POB)
gdf_pop_wgs84 = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[COL_LON], df[COL_LAT]),
    crs="EPSG:4326"
)
gdf_pop_local = gdf_pop_wgs84.to_crs(LOCAL_CRS)
total_poblacion = gdf_pop_local[COL_POB].sum()
print(f"Población total provincial: {total_poblacion:,.0f} hab")

bounds = gdf_pop_wgs84.total_bounds
sw = [bounds[1], bounds[0]]
ne = [bounds[3], bounds[2]]

# ------------------------------------------------------------
# 1. CARGA DEL GRAFO
# ------------------------------------------------------------
if os.path.exists(GRAPH_FILE):
    print("Cargando grafo guardado...")
    G = ox.load_graphml(GRAPH_FILE)
else:
    print("Descargando grafos locales...")
    graphs = []
    DIST = 80000
    for name, (lat, lon) in hospitals.items():
        print(f" → {name}")
        G_local = ox.graph_from_point((lat, lon), dist=DIST, network_type='drive')
        graphs.append(G_local)
    G = nx.compose_all(graphs)
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    ox.save_graphml(G, GRAPH_FILE)
    print("Grafo guardado.")
print(f"Grafo listo: {len(G.nodes)} nodos, {len(G.edges)} aristas")


# ------------------------------------------------------------
# 2. FUNCIÓN DE CÁLCULO DE ISÓCRONAS (se mantiene convex hull)
# ------------------------------------------------------------
from shapely.geometry import Point, MultiPoint
from shapely.ops import unary_union

def get_isochrones_for_hospital(G, center_coords, trip_times):
    """
    Calcula polígonos de isócronas para un hospital dado.
    
    Args:
        G: grafo de OSMnx (con travel_time en segundos)
        center_coords: tupla (lat, lon) del hospital
        trip_times: lista de tiempos en minutos
        
    Returns:
        dict {tiempo_en_minutos: polígono shapely}
    """
    import networkx as nx
    import geopandas as gpd

    # Convertir coordenadas hospital a nodo más cercano
    center_node = ox.nearest_nodes(G, X=center_coords[1], Y=center_coords[0])

    # Tiempo de viaje en segundos
    times_sec = [t*60 for t in trip_times]

    # Longitud de los edges para estimación de distancias
    travel_times = nx.single_source_dijkstra_path_length(G, center_node, weight='travel_time')

    # Crear GeoDataFrames de nodos
    nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
    nodes_proj = nodes.to_crs(LOCAL_CRS)

    isochrones = {}

    for t_sec, t_min in zip(times_sec, trip_times):
        # Nodos dentro del tiempo de viaje
        reachable_nodes = [n for n, tt in travel_times.items() if tt <= t_sec]
        if not reachable_nodes:
            continue
        gdf_nodes_proj = nodes_proj.loc[reachable_nodes]

        # Construir polígono de la isócrona
        multipoint = unary_union(gdf_nodes_proj.geometry)
        poly = multipoint.convex_hull
        if not poly.is_valid:
            poly = poly.buffer(0)  # limpieza geometría

        isochrones[t_min] = poly

    return isochrones

# ------------------------------------------------------------
# 3. CÁLCULO DE ISÓCRONAS
# ------------------------------------------------------------
print("\nCalculando isócronas individuales...")
isochrones_by_hospital = {}
all_polygons_by_time = {t: [] for t in trip_times}

for name, coords in hospitals.items():
    print(f"  {name}")
    polys = get_isochrones_for_hospital(G, coords, trip_times)
    isochrones_by_hospital[name] = polys
    for t, poly in polys.items():
        all_polygons_by_time[t].append(poly)

union_polygons_local = {}
union_polygons_wgs84 = {}
for t in trip_times:
    if all_polygons_by_time[t]:
        # Unir todos los polígonos en uno solo limpio
        from shapely.ops import unary_union
        cobertura_total = unary_union(all_polygons_by_time[t])
        cobertura_total = cobertura_total.simplify(50)  # opcional: simplifica geometría para render
        cobertura_total = cobertura_total.buffer(0)     # limpia geometría si hay errores
        
        union_polygons_local[t] = cobertura_total
        union_polygons_wgs84[t] = gpd.GeoSeries([cobertura_total], crs=LOCAL_CRS).to_crs("EPSG:4326").iloc[0]
        
        area_km2 = union_polygons_local[t].area / 1e6
        print(f"Cobertura combinada {t//60}h: área = {area_km2:.1f} km²")
    else:
        union_polygons_local[t] = None
        union_polygons_wgs84[t] = None        

# ------------------------------------------------------------
# 4. ESTADÍSTICAS POBLACIONALES DE COBERTURA
# ------------------------------------------------------------
stats_rows = []
for name, polys in isochrones_by_hospital.items():
    row = {"Hospital": name}
    for t in trip_times:
        if t in polys:
            pop = gdf_pop_local[gdf_pop_local.geometry.within(polys[t])][COL_POB].sum()
            row[f"Población_{t//60}h"] = int(pop)
        else:
            row[f"Población_{t//60}h"] = 0
    stats_rows.append(row)

df_hospital_stats = pd.DataFrame(stats_rows).set_index("Hospital")
print("\n=== Cobertura poblacional por hospital ===")
print(df_hospital_stats.to_string())

union_coverage_pop = {}
for t in trip_times:
    if union_polygons_local[t] is not None:
        covered = gdf_pop_local[gdf_pop_local.geometry.within(union_polygons_local[t])]
        union_coverage_pop[t] = covered[COL_POB].sum()
    else:
        union_coverage_pop[t] = 0

total_cubierta_4h = union_coverage_pop[240]
porcentaje_cubierto = (total_cubierta_4h / total_poblacion) * 100
area_total_km2 = union_polygons_local[240].area / 1e6 if union_polygons_local[240] else 0

print("\n=== Resumen red completa ===")
print(f"Población cubierta (≤4h): {total_cubierta_4h:,.0f} ({porcentaje_cubierto:.1f}%)")
print(f"Área cubierta: {area_total_km2:,.0f} km²")

# Guardar CSV
df_hospital_stats.to_csv(os.path.join(OUT_DIR_HTML, "cobertura_por_hospital.csv"))
summary_df = pd.DataFrame({
    "Indicador": ["Población total", "Cubierta (≤4h)", "Porcentaje", "Área (km²)"],
    "Valor": [f"{total_poblacion:,.0f}", f"{total_cubierta_4h:,.0f}", f"{porcentaje_cubierto:.1f}%", f"{area_total_km2:,.0f}"]
})
summary_df.to_csv(os.path.join(OUT_DIR_HTML, "resumen_red_completa.csv"), index=False)

# ------------------------------------------------------------
# 5. CARGA DE INDICADORES DE RIESGO (CSVs) y CÁLCULO DE RISK SCORE
# ------------------------------------------------------------
print("\nCargando indicadores de riesgo desde CSVs...")
ruta2 = os.path.join(BASE_DIR, "data", "indicadores_2.csv")
ruta3 = os.path.join(BASE_DIR, "data", "indicadores_3.csv")
ruta4 = os.path.join(BASE_DIR, "data", "indicadores_4.csv")

def load_and_clean_csv(path):
    df = pd.read_csv(path, encoding='utf-8', sep=';')
    df.columns = (df.columns.str.strip()
                  .str.lower()
                  .str.normalize('NFKD')
                  .str.encode('ascii', errors='ignore')
                  .str.decode('utf-8')
                  .str.replace(' ', '_')
                  .str.replace('á','a').str.replace('é','e').str.replace('í','i')
                  .str.replace('ó','o').str.replace('ú','u').str.replace('ñ','n')
                  .str.replace(r'\.$','', regex=True))
    # Identificar columna de código de radio
    codigo_cols = [col for col in df.columns if 'codigo' in col and 'radio' in col]
    if len(codigo_cols) == 0:
        raise KeyError("No se encontró columna de código de radio")
    codigo_col = codigo_cols[0]
    df.rename(columns={codigo_col: 'codigo_radio'}, inplace=True)
    # Convertir a numérico
    for col in df.columns:
        if col != 'codigo_radio':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.loc[:, ~df.columns.duplicated()]
    return df

df2 = load_and_clean_csv(ruta2)
df3 = load_and_clean_csv(ruta3)
df4 = load_and_clean_csv(ruta4)

# Unir los CSVs
gdf_temp = df2.merge(df3, on='codigo_radio', how='left', suffixes=('_2', '_3'))
gdf_indicadores = gdf_temp.merge(df4, on='codigo_radio', how='left', suffixes=('', '_4'))

# Asegurar que gdf_pop_local tenga 'codigo_radio'
if 'codigo_radio' not in gdf_pop_local.columns:
    for col in gdf_pop_local.columns:
        if 'radio' in col.lower() or 'codigo' in col.lower():
            gdf_pop_local = gdf_pop_local.rename(columns={col: 'codigo_radio'})
            break
    else:
        if 'codigo_de_radio' in df.columns:
            gdf_pop_local['codigo_radio'] = df['codigo_de_radio'].values
        elif 'codigo_radio' in df.columns:
            gdf_pop_local['codigo_radio'] = df['codigo_radio'].values

# Unir indicadores con coordenadas
gdf_modelo = gdf_indicadores.merge(gdf_pop_local[['codigo_radio', 'geometry']], on='codigo_radio', how='left')
gdf_modelo = gpd.GeoDataFrame(gdf_modelo, geometry='geometry', crs=gdf_pop_local.crs)

# Columnas de riesgo base
cols_riesgo_base = [
    'hogares_con_hacinamiento_critico_(mas_de_3_personas_por_cuarto)',
    'hogares_sin_cloaca',
    'hogares_sin_agua_para_beber_y_cocinar_proveniente_de_red_publica',
    'hogares_con_computadora',
    'hogares_con_telefono_celular',
    'poblacion_de_70_anos_y_mas',
    'solo_salud_publica',
    'poblacion_de_18_y_mas_con_primaria_incompleta_o_menos'
]

# Buscar columnas con posibles sufijos
for col in cols_riesgo_base:
    if col not in gdf_modelo.columns:
        for suf in ['_3', '_4', '_2']:
            col_suf = f"{col}{suf}"
            if col_suf in gdf_modelo.columns:
                gdf_modelo[col] = gdf_modelo[col_suf]
                break

existing_risk_cols = [col for col in cols_riesgo_base if col in gdf_modelo.columns]
missing = [col for col in cols_riesgo_base if col not in existing_risk_cols]
if missing:
    print("⚠ Columnas de riesgo no encontradas (serán ignoradas):", missing)

# Invertir columnas donde mayor valor = menor riesgo
if 'hogares_con_computadora' in existing_risk_cols:
    gdf_modelo['hogares_con_computadora_inv'] = 1 - gdf_modelo['hogares_con_computadora'].fillna(0)/100
    existing_risk_cols.append('hogares_con_computadora_inv')
    existing_risk_cols.remove('hogares_con_computadora')
if 'hogares_con_telefono_celular' in existing_risk_cols:
    gdf_modelo['hogares_con_telefono_celular_inv'] = 1 - gdf_modelo['hogares_con_telefono_celular'].fillna(0)/100
    existing_risk_cols.append('hogares_con_telefono_celular_inv')
    existing_risk_cols.remove('hogares_con_telefono_celular')

# Calcular risk_score
if len(existing_risk_cols) > 0:
    scaler = MinMaxScaler()
    X = gdf_modelo[existing_risk_cols].fillna(0)
    risk_normalized = scaler.fit_transform(X)
    gdf_modelo['risk_score_norm'] = risk_normalized.mean(axis=1)
    print("✅ Risk_score calculado correctamente.")
else:
    print("❌ No hay columnas de riesgo, no se puede calcular risk_score.")
    gdf_modelo['risk_score_norm'] = np.nan

# ------------------------------------------------------------
# 6. FUNCIÓN PARA GUARDAR MAPAS (HTML + PNG)
# ------------------------------------------------------------
def html_to_png(html_path, png_path, width=1200, height=900):
    if not SELENIUM_AVAILABLE:
        return
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument(f"--window-size={width},{height}")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        driver.get("file://" + os.path.abspath(html_path))
        time.sleep(4)
        driver.save_screenshot(png_path)
        driver.quit()
        print(f"   📸 PNG guardado: {png_path}")
    except Exception as e:
        print(f"   ⚠️ PNG no generado: {e}")

def save_map(m, name, also_png=True):
    html_path = os.path.join(OUT_DIR_HTML, f"{name}.html")
    m.save(html_path)
    print(f"✅ Mapa HTML: {html_path}")
    if also_png:
        png_path = os.path.join(OUT_DIR_PNG, f"{name}.png")
        html_to_png(html_path, png_path)

# ------------------------------------------------------------
# 7. MAPA 1: COBERTURA COMBINADA
# ------------------------------------------------------------
m1 = folium.Map(tiles='CartoDB positron')
m1.fit_bounds([sw, ne])

heat_data = [[row.geometry.y, row.geometry.x, row[COL_POB]] for _, row in gdf_pop_wgs84.iterrows()]
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m1)

for i, t in enumerate(trip_times):
    if union_polygons_wgs84[t] is None:
        continue
    pop = union_coverage_pop[t]
    folium.GeoJson(
        union_polygons_wgs84[t].__geo_interface__,
        style_function=lambda x, c=iso_colors_hex[i]: {'fillColor': c, 'color': c, 'weight': 1.5, 'fillOpacity': 0.2},
        tooltip=f"{t//60}h: {pop:,.0f} personas",
        name=iso_names[i]
    ).add_to(m1)

for name, (lat, lon) in hospitals.items():
    folium.Marker(location=[lat, lon], popup=name, tooltip=name,
                  icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m1)

legend1_html = f"""
<div style="position: fixed; bottom: 30px; left: 30px; width: 300px; 
     background-color: white; border:2px solid grey; border-radius:8px; 
     padding: 10px; font-size: 13px; z-index: 9999; 
     box-shadow: 3px 3px 6px rgba(0,0,0,0.3);">
<h4 style="margin-top:0; text-align:center;">Isocronas y cobertura poblacional</h4>
<b>🚗 Tiempo de viaje (automóvil):</b><br>
<span style="display:inline-block; width:20px; height:12px; background:#ff4d4d;"></span> 1 hora<br>
<span style="display:inline-block; width:20px; height:12px; background:#ffaa00;"></span> 2 horas<br>
<span style="display:inline-block; width:20px; height:12px; background:#4dff4d;"></span> 4 horas<br>
<hr>
<b>🔥 Mapa de calor (población):</b><br>
<div style="width:100%; height:20px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
<span style="float:left;">Menor población</span><span style="float:right;">Mayor población</span><br clear="all">
<hr>
<b>🏥 Hospitales:</b> Icono “+” negro<br>
<b>📊 Población total cubierta (≤4h):</b> {total_cubierta_4h:,.0f} personas ({porcentaje_cubierto:.1f}%)<br>
<small>Los tooltips muestran población dentro de cada isócrona.</small>
</div>
"""
m1.get_root().html.add_child(folium.Element(legend1_html))
folium.LayerControl().add_to(m1)
save_map(m1, "mapa_cobertura_combinada", also_png=True)

# ------------------------------------------------------------
# 8. MAPA 2: SOLO DENSIDAD POBLACIONAL
# ------------------------------------------------------------
m2 = folium.Map(tiles='CartoDB positron')
m2.fit_bounds([sw, ne])
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m2)

legend2_html = f"""
<div style="position: fixed; bottom: 30px; left: 30px; width: 280px; 
     background-color: white; border:2px solid grey; border-radius:8px; 
     padding: 10px; font-size: 13px; z-index: 9999;">
<h4 style="margin-top:0;">Densidad poblacional - Salta (Censo 2022)</h4>
<div style="width:100%; height:20px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
<span style="float:left;">Baja densidad</span><span style="float:right;">Alta densidad</span><br clear="all">
<hr>
<b>Población total provincial:</b> {total_poblacion:,.0f} personas<br>
<small>Cada radio censal ponderado por su población.</small>
</div>
"""
m2.get_root().html.add_child(folium.Element(legend2_html))
save_map(m2, "mapa_densidad_poblacional", also_png=True)

# ------------------------------------------------------------
# 9. MAPAS INDIVIDUALES POR HOSPITAL
# ------------------------------------------------------------
for hosp_name, coords in hospitals.items():
    m = folium.Map(tiles='CartoDB positron')
    m.fit_bounds([sw, ne])
    HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m)

    polys_local = isochrones_by_hospital.get(hosp_name, {})
    for i, t in enumerate(trip_times):
        if t not in polys_local:
            continue
        poly_wgs84 = gpd.GeoSeries([polys_local[t]], crs=LOCAL_CRS).to_crs("EPSG:4326").iloc[0]
        pop_in = gdf_pop_local[gdf_pop_local.geometry.within(polys_local[t])][COL_POB].sum()
        folium.GeoJson(
            poly_wgs84.__geo_interface__,
            style_function=lambda x, c=iso_colors_hex[i]: {'fillColor': c, 'color': c, 'weight': 1.5, 'fillOpacity': 0.2},
            tooltip=f"{t//60}h: {pop_in:,.0f} personas",
            name=iso_names[i]
        ).add_to(m)

    folium.Marker(location=coords, popup=hosp_name, tooltip=hosp_name,
                  icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m)

    legend_hosp_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: 280px; 
         background-color: white; border:2px solid grey; border-radius:8px; padding: 10px; font-size: 13px; z-index: 9999;">
    <b>{hosp_name}</b><br>
    <span style="background:#ff4d4d; width:20px;height:12px;display:inline-block;"></span> 1h &nbsp;
    <span style="background:#ffaa00; width:20px;height:12px;display:inline-block;"></span> 2h &nbsp;
    <span style="background:#4dff4d; width:20px;height:12px;display:inline-block;"></span> 4h<br>
    <div style="width:100%; height:15px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
    <small>Mapa de calor: densidad poblacional. Tooltips muestran población por isócrona.</small>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_hosp_html))
    folium.LayerControl().add_to(m)
    save_map(m, f"mapa_{hosp_name.replace(' ', '_')}", also_png=True)

# ------------------------------------------------------------
# 10. MAPA DE RIESGO FUERA DE COBERTURA (2h) - coloreado por riesgo
# ------------------------------------------------------------
print("\n--- Generando mapa de riesgo fuera de cobertura 2h ---")


tiempo_cluster = 120  # minutos
if union_polygons_local.get(tiempo_cluster) is None:
    print(f"No hay polígono de cobertura para {tiempo_cluster//60}h.")
else:
    poly_local = union_polygons_local[tiempo_cluster]
    fuera = gdf_pop_local[~gdf_pop_local.geometry.within(poly_local)].copy()  # .copy() evita warnings
    print(f"Radios fuera de cobertura: {len(fuera)} de {len(gdf_pop_local)}")
    print(f"Población fuera: {fuera[COL_POB].sum():,.0f} hab")

    if 'codigo_radio' not in fuera.columns:
        for col in fuera.columns:
            if 'radio' in col.lower() or 'codigo' in col.lower():
                fuera = fuera.rename(columns={col: 'codigo_radio'})
                print(f"Columna '{col}' renombrada a 'codigo_radio'")
                break
        else:
            raise KeyError("No se encontró columna de código de radio en 'fuera'.")

    # Convertir a string y limpiar espacios para el merge
    gdf_modelo['codigo_radio'] = gdf_modelo['codigo_radio'].astype(str).str.strip()
    fuera['codigo_radio'] = fuera['codigo_radio'].astype(str).str.strip()

    print(f"Radios fuera ANTES del merge: {len(fuera)}")
    print(f"Radios con riesgo en gdf_modelo: {len(gdf_modelo[gdf_modelo['risk_score_norm'].notna()])}")

    if 'risk_score_norm' not in fuera.columns:
        if 'codigo_radio' in gdf_modelo.columns and 'risk_score_norm' in gdf_modelo.columns:
            fuera = fuera.merge(gdf_modelo[['codigo_radio', 'risk_score_norm']], 
                                on='codigo_radio', how='left')
            print(f"Radios DESPUÉS del merge: {len(fuera)}")
            print(f"NaN en risk_score_norm después del merge: {fuera['risk_score_norm'].isna().sum()}")
        else:
            raise KeyError("Faltan columnas en gdf_modelo para agregar riesgo.")

    print(f"Radios con riesgo válido ANTES de dropna: {len(fuera)}")
    fuera = fuera.dropna(subset=['risk_score_norm'])
    print(f"Radios DESPUÉS del dropna (los que se dibujarán): {len(fuera)}")

    if len(fuera) == 0:
        print("⚠ ADVERTENCIA: No hay radios fuera de cobertura con riesgo válido. Verifica el merge.")
    else:
        print(f"Rango de riesgo: min={fuera['risk_score_norm'].min():.3f}, max={fuera['risk_score_norm'].max():.3f}, todos iguales? {fuera['risk_score_norm'].nunique() == 1}")

    # ------------------- 3. CONVERTIR A WGS84 PARA FOLIUM -------------------
    fuera_wgs84 = fuera.to_crs(epsg=4326)

    # ------------------- 4. GENERAR MAPA -------------------
    m_fuera = folium.Map(location=[-24.782, -65.423], zoom_start=8, tiles='CartoDB positron')

    # Polígono de cobertura 2h (ya en WGS84)
    if union_polygons_wgs84.get(tiempo_cluster):
        folium.GeoJson(
            union_polygons_wgs84[tiempo_cluster].__geo_interface__,
            style_function=lambda x: {'fillColor': '#4dff4d', 'color': '#4dff4d', 'weight': 1, 'fillOpacity': 0.1},
            tooltip="Cobertura 2h"
        ).add_to(m_fuera)

    # Escala de riesgo (amarillo → rojo)
    try:
        from branca.colormap import linear
        risk_colormap = linear.YlOrRd_09.scale(0, 1)
        risk_colormap.caption = 'Riesgo normalizado'
    except (ImportError, NameError):
        def risk_colormap(x):
            r = 255
            g = int(255 * (1 - x))
            b = 0
            return f'#{r:02x}{g:02x}{b:02x}'
        print("Usando colormap manual (fallback)")

    if len(fuera) > 0:
        for idx, row in fuera_wgs84.iterrows():
            try:
                color = risk_colormap(row['risk_score_norm'])
            except Exception:
                color = 'red'
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=4,
                color=color,
                fill=True,
                fill_opacity=0.8,
                popup=f"Radio: {row['codigo_radio']}<br>Población: {row[COL_POB]:,.0f}<br>Riesgo: {row['risk_score_norm']:.2f}"
            ).add_to(m_fuera)

        if hasattr(risk_colormap, 'add_to'):
            risk_colormap.add_to(m_fuera)

    # Leyenda personalizada
    riesgo_promedio = fuera['risk_score_norm'].mean() if len(fuera) > 0 else 0
    leyenda_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: 300px; background-color: white; border:2px solid grey; border-radius:8px; padding: 10px; font-size: 12px; z-index: 9999;">
    <b>Radios fuera de cobertura 2h</b><br>
    Población total: {fuera[COL_POB].sum():,.0f} hab<br>
    N° de radios: {len(fuera)}<br>
    Riesgo promedio: {riesgo_promedio:.2f}<br>
    <hr>
    <span style="background:#ffffb2; width:20px;height:12px;display:inline-block;"></span> Bajo riesgo<br>
    <span style="background:#fd8d3c; width:20px;height:12px;display:inline-block;"></span> Medio riesgo<br>
    <span style="background:#bd0026; width:20px;height:12px;display:inline-block;"></span> Alto riesgo<br>
    <small>El color de cada punto indica su nivel de riesgo (escala amarillo-rojo).</small>
    </div>
    """
    m_fuera.get_root().html.add_child(folium.Element(leyenda_html))

    map_file = os.path.join(OUT_DIR_HTML, "mapa_riesgo_fuera_cobertura_2h.html")
    m_fuera.save(map_file)
    print(f"✅ Mapa de riesgo fuera de cobertura guardado: {map_file}")

    if SELENIUM_AVAILABLE:
        png_path = os.path.join(OUT_DIR_PNG, "mapa_riesgo_fuera_cobertura_2h.png")
        html_to_png(map_file, png_path) 

# ------------------------------------------------------------
# 11. MAPA DE COBERTURA 4h con heatmap acotado
# ------------------------------------------------------------
if union_polygons_wgs84[240] is not None:
    m_punto2 = folium.Map(tiles='CartoDB positron')
    bounds_4h = union_polygons_wgs84[240].bounds
    m_punto2.fit_bounds([[bounds_4h[1], bounds_4h[0]], [bounds_4h[3], bounds_4h[2]]])

    gdf_pop_4h = gdf_pop_local[gdf_pop_local.geometry.within(union_polygons_local[240])].copy()
    gdf_pop_4h_wgs84 = gdf_pop_4h.to_crs("EPSG:4326")
    heat_data_4h = [[row.geometry.y, row.geometry.x, row[COL_POB]] for _, row in gdf_pop_4h_wgs84.iterrows()]
    HeatMap(heat_data_4h, radius=15, blur=10, min_opacity=0.3,
            name="Densidad poblacional (dentro de 4h)").add_to(m_punto2)

    folium.GeoJson(
        union_polygons_wgs84[240].__geo_interface__,
        style_function=lambda x: {'fillColor': '#4dff4d', 'color': '#4dff4d', 'weight': 2, 'fillOpacity': 0.2},
        tooltip=f"4 horas: {union_coverage_pop[240]:,.0f} personas",
        name="Cobertura 4h"
    ).add_to(m_punto2)

    for name, (lat, lon) in hospitals.items():
        folium.Marker(location=[lat, lon], popup=name, tooltip=name,
                      icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m_punto2)

    legend_punto2 = f"""
    <div style="position: fixed; bottom: 30px; right: 30px; width: 280px; 
         background-color: white; border:2px solid grey; border-radius:8px; 
         padding: 10px; font-size: 12px; z-index: 9999; 
         box-shadow: 3px 3px 6px rgba(0,0,0,0.3);">
    <b>🗺️ Cobertura de 4 horas – Red de ACV</b><br>
    <span style="background:#4dff4d; width:20px;height:12px;display:inline-block;"></span> Área alcanzable en ≤4h<br>
    <div style="width:100%; height:15px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
    <span style="float:left;">Baja densidad</span><span style="float:right;">Alta densidad</span><br clear="all">
    <b>🏥 Hospitales:</b> + negro<br>
    <b>👥 Población dentro del área:</b> {union_coverage_pop[240]:,.0f} hab ({porcentaje_cubierto:.1f}%)<br>
    <small>El mapa de calor solo muestra población dentro de la cobertura de 4h.</small>
    </div>
    """
    m_punto2.get_root().html.add_child(folium.Element(legend_punto2))
    folium.LayerControl().add_to(m_punto2)
    save_map(m_punto2, "mapa_cobertura_4h_con_heatmap_acotado", also_png=True)

# ------------------------------------------------------------
# 12. MAPAS DE COBERTURA POR TIEMPO (1h, 2h, 4h)
# ------------------------------------------------------------
for i, t in enumerate(trip_times):
    if union_polygons_wgs84[t] is None:
        continue
    m = folium.Map(tiles='CartoDB positron')
    m.fit_bounds([sw, ne])
    HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3, name="Densidad poblacional").add_to(m)
    pop_in = union_coverage_pop[t]
    folium.GeoJson(
        union_polygons_wgs84[t].__geo_interface__,
        style_function=lambda x, c=iso_colors_hex[i]: {'fillColor': c, 'color': c, 'weight': 2, 'fillOpacity': 0.3},
        tooltip=f"{t//60}h: {pop_in:,.0f} personas",
        name=f"{t//60}h"
    ).add_to(m)
    for name, (lat, lon) in hospitals.items():
        folium.Marker(location=[lat, lon], popup=name, icon=folium.Icon(color='black', icon='plus', prefix='fa')).add_to(m)

    legend_time_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: 260px; 
         background-color: white; border:2px solid grey; border-radius:8px; 
         padding: 10px; font-size: 13px; z-index: 9999;">
    <b>Cobertura de {t//60} hora(s)</b><br>
    <span style="background:{iso_colors_hex[i]}; width:20px;height:12px;display:inline-block;"></span> Isócrona {t//60}h<br>
    <b>Población dentro del área:</b> {pop_in:,.0f} hab<br>
    <div style="width:100%; height:15px; background: linear-gradient(to right, #00FF00, #FFFF00, #FF7F00, #FF0000); margin:5px 0;"></div>
    <small>Mapa de calor: densidad poblacional. Los tooltips muestran la población exacta.</small>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_time_html))
    save_map(m, f"mapa_cobertura_{t//60}h", also_png=True)

print("\n" + "="*60)
print("✅ ¡PROCESO COMPLETADO EXITOSAMENTE!")
print(f"📁 HTMLs interactivos: {os.path.abspath(OUT_DIR_HTML)}")
print(f"🖼️  Imágenes PNG: {OUT_DIR_PNG}")
print("="*60)